### Derived property filtering demo

In this demo we will showcase the derived property filtering capabilities of LUSID.

In [ ]:
!pip3 install -U lusid-sdk finbourne-sdk-utils

In [ ]:
# Set up LUSID
import os
import pandas as pd
import json
import uuid
import time
from IPython.core.display import HTML
import logging
logging.basicConfig(level=logging.INFO)

import lusid as lss
import lusid.api as la
import lusid.models as lm

from lusidjam import RefreshingToken
from lusid.extensions import (
    SyncApiClientFactory,
    ArgsConfigurationLoader,
    EnvironmentVariablesConfigurationLoader,
    SecretsFileConfigurationLoader
)
from finbourne_sdk_utils.pandas_utils.lusid_pandas import lusid_response_to_data_frame
from finbourne_sdk_utils.jupyter_tools import StopExecution
from finbourne_sdk_utils.lpt.lpt import to_date

# Set pandas display options
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.options.display.float_format = "{:,.2f}".format
#display(HTML("<style>.container { width:90% !important; }</style>"))

# Authenticate to SDK
# Run the Notebook in Jupyterhub for your LUSID domain and authenticate automatically
secrets_path = os.getenv("FBN_SECRETS_PATH")
# Run the Notebook locally using a secrets file (see https://support.lusid.com/knowledgebase/article/KA-01663)
if secrets_path is None:
    secrets_path = os.path.join(os.path.dirname(os.getcwd()), "secrets.json")

# Initiate an API Factory which is the client side object for interacting with LUSID APIs
config_loaders=[
    ArgsConfigurationLoader(access_token = RefreshingToken(), app_name = "LusidJupyterNotebook"),
    EnvironmentVariablesConfigurationLoader(),
    SecretsFileConfigurationLoader(secrets_path)]
api_factory = SyncApiClientFactory(config_loaders=config_loaders)
    
# Confirm success by printing SDK version
api_status = pd.DataFrame(api_factory.build(lu.ApplicationMetadataApi).get_lusid_versions().to_dict())
display(api_status)

## Global variables

In [ ]:
test_scope = str(uuid.uuid4())

## 1. Populating LUSID with some instruments

First, we'll create some instruments in our domain. We'll use the instrument names as part of our derivation formula in the next step.

In [ ]:
instruments_api = api_factory.build(lu.InstrumentsApi) 

# Upsert 5 instruments
request_dictionary = {
    i: lu.InstrumentDefinition(
        name=f"Instrument for derived property testing-{i}",
        identifiers={
            "ClientInternal": lu.InstrumentIdValue(value=f"Client internal value - {i}")
        }
    )
    for i in range(5)
}

upsert_instruments_response = instruments_api.upsert_instruments(
    request_body=request_dictionary,
    scope=test_scope)

upsert_instruments_response_df = lusid_response_to_data_frame(list(upsert_instruments_response.values.values()))
display(upsert_instruments_response_df[["name"]])

## 2. Setting up a new derived property definition
In this step, we will create an instrument derived property definition which derives from each instrument's name.

In [ ]:
property_definitions_api = api_factory.build(lu.PropertyDefinitionsApi)

derived_property_definition = property_definitions_api.create_derived_property_definition(
    create_derived_property_definition_request=lu.CreateDerivedPropertyDefinitionRequest(
        domain="Instrument",
        code="DemoCode",
        scope=test_scope,
        displayName="Demo",
        dataTypeId=lu.ResourceId(scope="system", code="string"),
        propertyDescription="Demo",
        derivationFormula="concat('Instrument name: ', Name)",
        isFilterable=True))
    
print(f"Derived property definition key: {derived_property_definition.key}")
print(f"Derived property definition derivation formula: {derived_property_definition.derivation_formula}")


## 3. Filtering instruments based on their derived property values
Finally, we'll list instruments with a filter on our expected derived property values.
We have a retry mechanism to wait until the background derived property calculation is complete.

In [ ]:
filter = f"Properties[{derived_property_definition.key}] startswith 'Instrument name: Instrument for derived property testing'"

retry_count = 0

while retry_count < 5:
    try:
        instruments = instruments_api.list_instruments(
            filter=filter,
            scope=test_scope,
            limit=5000)
        retry_count += 1
    except lu.ApiException as e:
        if json.loads(e.body)["name"] == "DerivedPropertyCalculationNotComplete":
            print("Derived property not available for filtering yet. Retrying in 2s.") 
            time.sleep(2)
        else:
            raise

instruments_df = lusid_response_to_data_frame(list(instruments.values))
display(instruments_df[["name", "properties.0.value.labelValue"]])
